In [1]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import ShuffleSplit
import pandas as pd
import numpy as np
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import auc
from src.metrics import dice_coef, rc_curve
from src.confidence import (SoftDice, Lesion_Load, MeanMaxConfidence,ExpectedEntropy,Median_min_max,Quantile_thresh, Patch_Level_Aggregation)
from src.utils import  write_aurc_curves, thresh_alpha, get_tuning_size

In [2]:
task ='polyp'
thresh = 0.5

In [3]:
with open(f'./data/{task}_features_dict.pkl', 'rb') as file:
    # Save the list to the file
    features_dict =  pickle.load(file)

In [4]:
with open(f'./data/{task}_u_threshs.pkl', 'rb') as file:
    # Load the  list to the file
    u_threshs = pickle.load(file)

In [5]:
with open(f'./data/{task}_targets.pkl', 'rb') as file:
    # Save the list to the file
    targets =  pickle.load(file)

In [6]:
if 'mswml' not in task:
    data = np.load(f'./data/{task}.npz')
    y = data['y']
    p_hat = data['p_hat']
    try:
        y_hat = data['y_hat'] 
    except KeyError:
        y_hat = p_hat >= thresh
elif 'ood' in task:
    p_hat = pickle.load(open('./data/mswml/dev_out_predictions.pkl', 'rb'))
    y_hat = [p_hat_i >= thresh for p_hat_i in p_hat] 
    y = pickle.load(open('./data/mswml/dev_out_gt.pkl', 'rb'))
else:
    p_hat = pickle.load(open('./data/mswml/eval_in_predictions.pkl', 'rb'))
    y_hat = [p_hat_i >= thresh for p_hat_i in p_hat] 
    y = pickle.load(open('./data/mswml/eval_in_gt.pkl', 'rb'))

In [7]:
def replace_nested_nans(value):
    if isinstance(value, np.ndarray):  # Check if the value is a numpy array
        return np.where(np.isnan(value), 0, value)  # Replace nan with 0 in the array
    elif pd.isna(value):  # If it's a scalar NaN value
        return 0
    else:
        return value

In [8]:
# For each key in features_dict convert the list of features to a DataFrame
df_features_dict = {}
for key, value in features_dict.items():
    df_features_dict[key] = pd.DataFrame(value).map(replace_nested_nans)


df_targets = pd.Series(targets)

In [9]:
feature_SDC=[]
sdc = SoftDice(threshold_lim= thresh)
#for p_hat_i in p_hat:
#    feature_SDC.append(sdc(p_hat_i))
#If y_hat is not provided, uncomment the lines above and comment the two lines below
for p_hat_i, y_hat_i in zip(p_hat,y_hat): 
    feature_SDC.append(sdc(p_hat_i, y_hat=y_hat_i)) 
feature_SDC = np.array(feature_SDC)

In [10]:
def aurc_best_patch_level(patch_sizes,p_hat_test, y_hat_test,y_test, threshold=thresh):
    aurcs = []
    best_patches = []
    dice_scores = np.stack([dice_coef(y_hat_test[i], y_test[i],threshold=threshold) for i in range(len(y_test))])
    dice_errors = 1 - dice_scores
    for patch_size in patch_sizes:
        patch_size = int(patch_size)
        confidence_metric =  Patch_Level_Aggregation(patch_size=patch_size)
        confidence = [confidence_metric(p_hat_i) for p_hat_i in p_hat_test]
        coverage, risk, _ = rc_curve(confidence, dice_errors)
        aurc = auc(coverage, risk)
        aurcs.append(aurc)
        best_patches.append(patch_size)
    return best_patches[np.argmin(aurcs)]

In [11]:
import os

n = 50
tuning_size_arrays = get_tuning_size(len(y))  # Tuning sizes
patch_sizes = np.concatenate((np.arange(1, 10, 1), np.arange(10, 201, 5)), axis=0)

mean_aurcs_file = f'df_aurcs_mean_final_{task}.pkl'
std_aurcs_file = f'df_aurcs_std_final_{task}.pkl'

if os.path.exists(mean_aurcs_file) and os.path.exists(std_aurcs_file):
    df_aurcs_mean = pd.read_pickle(mean_aurcs_file)
    df_std_dev = pd.read_pickle(std_aurcs_file)
    mean_aurcs = [df_aurcs_mean.iloc[i] for i in range(len(df_aurcs_mean))]
    std_aurcs = [df_std_dev.iloc[i] for i in range(len(df_std_dev))]
    completed = len(mean_aurcs)
    count = completed * n
    print(f"Resuming from tuning_size index {completed}, count={count}")
else:
    mean_aurcs = []
    std_aurcs = []
    completed = 0
    count = 0

for idx, tuning_size in enumerate(tuning_size_arrays):
    if idx < completed:
        continue
    aurcs = []
    for i in range(n):
        shuffle_split = ShuffleSplit(n_splits=1, train_size=tuning_size, random_state=i)
        for train_index, test_index in shuffle_split.split(df_targets):
            df_features = df_features_dict[u_threshs[count]]
            Xf_train, Xf_test = df_features.iloc[train_index], df_features.iloc[test_index]
            yf_train, yf_test = df_targets.iloc[train_index], df_targets.iloc[test_index]
            p_hat_train = [p_hat[i] for i in train_index]
            p_hat_test = [p_hat[i] for i in test_index]
            y_hat_train = [y_hat[i] for i in train_index]
            y_hat_test = [y_hat[i] for i in test_index]
            y_train = [y[i] for i in train_index]
            y_test = [y[i] for i in test_index]
            feature_SDC_train, feature_SDC_test = feature_SDC[train_index], feature_SDC[test_index]


            regressor_j = RandomForestRegressor(n_estimators=100, random_state=42, criterion='squared_error',max_depth=None)
            regressor_j.fit(Xf_train, yf_train)
             # Predict on the test set
            yrf_j_pred = regressor_j.predict(Xf_test)

            
            df_1_train = pd.DataFrame({
            'SDC': feature_SDC_train,
            #'PLL': feature_PLL_train
            })

            df_1_test = pd.DataFrame({
            'SDC': feature_SDC_test,
            #'PLL': feature_PLL_test
            })

            df_2_train = pd.concat([df_1_train.reset_index(drop=True), Xf_train.reset_index(drop=True)], axis=1)
            df_2_test =  pd.concat([df_1_test.reset_index(drop=True), Xf_test.reset_index(drop=True)], axis=1)
            

            
            regressor_4 = RandomForestRegressor(n_estimators=100, random_state=42, criterion='squared_error',max_depth=None)
            regressor_4.fit(df_2_train, yf_train)
            yrf_4_pred = regressor_4.predict(df_2_test)
           
            threshold_alpha = thresh_alpha(p_hat_train,threshold = thresh, y_hat=y_hat_train) 
            best_patch_size = aurc_best_patch_level(patch_sizes,p_hat_train, y_hat_train,y_train)
            confidence_metrics = {
                'MSP': MeanMaxConfidence(),
                'Neg.Entropy': ExpectedEntropy(),
                'Median_min_max':Median_min_max(),
                'Quantile_thresh':Quantile_thresh(threshold_alpha),
                'PLA':Patch_Level_Aggregation(patch_size=best_patch_size)
            }
            confidence_metric_sdc = SoftDice(thresh)
            # Preallocate outputs
            confidences = {name: [] for name in confidence_metrics}
            confidences['SDC'] = []
            
            # Single pass over the batch
            for p_hat_i,y_hat_i in zip(p_hat_test, y_hat_test):
                # Compute all per-prediction metrics in one go
                for name, metric in confidence_metrics.items():
                    confidences[name].append(metric(p_hat_i))
                # SDC needs y_hat
                confidences['SDC'].append(confidence_metric_sdc(p_hat_i, y_hat=y_hat_i))
    
            confidences['AEF'] = yrf_j_pred.tolist()
            confidences['AEF+SDC'] =yrf_4_pred.tolist()

            dice_scores = np.stack([dice_coef(y_hat_test[i], y_test[i],threshold=thresh) for i in range(len(y_test))])
            dice_errors = 1 - dice_scores

            aurcs.append(write_aurc_curves(confidences, dice_errors))

            count+=1
 
    mean_aurcs.append(pd.DataFrame(aurcs).mean())
    std_aurcs.append(pd.DataFrame(aurcs).std())
    df_aurcs_mean = pd.DataFrame(mean_aurcs) 
    df_std_dev = pd.DataFrame(std_aurcs)
    df_aurcs_mean.to_pickle(mean_aurcs_file)
    df_std_dev.to_pickle(std_aurcs_file)
    